# Stationarity

ADF and KPSS on price, log price and log returns, plus the same tests re-run within each year.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.research` and read back from
`results/statistical_research/<timeframe>/`. Generate those first:

```
xq research-summary --timeframe 5m
xq research-summary --all
```

Nothing here is a trading signal. No transaction costs are applied anywhere.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.research import load_research_config

config = load_config(ROOT / "config" / "data.yaml")
research = load_research_config(ROOT / "config" / "research.yaml")

TIMEFRAME = "5m"          # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"time basis : {research.intraday.time_basis}")
print(f"results    : {research.results_path}")

In [ ]:
import json

results = research.results_dir(TIMEFRAME)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME}. Run:  xq research-summary --timeframe {TIMEFRAME}"
    )

def table(name):
    """Load a previously-generated research table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
print(f"generated : {summary['provenance']['generated_utc']}")
print(f"span      : {summary['bars']['first_timestamp']} -> {summary['bars']['last_timestamp']}")
print(f"bars      : {summary['bars']['count']:,}")

## ADF and KPSS

Their nulls are opposites:

- **ADF** H0 = unit root (non-stationary). Small p rejects the unit root.
- **KPSS** H0 = stationary. Small p rejects stationarity.

Agreement between them is what carries information.

In [ ]:
stat = json.loads((results / "stationarity.json").read_text(encoding="utf-8"))
for entry in stat["series"]:
    print(f"=== {entry['series_name']}  (n={entry['observations']:,}) ===")
    for name in ("adf", "kpss"):
        test = entry.get(name)
        if test:
            print(f"  {name.upper():5} statistic={test['statistic']:10.4f}  "
                  f"p={test['p_value']:.4g}  rejects={test['reject_at_5pct']}")
    print(f"  VERDICT: {entry['verdict']}")
    print(f"  {entry['interpretation']}")
    print()

## Segmented by year

A verdict that changes between years is evidence of structural change.

In [ ]:
table('stationarity_segments')